# Предрасчёт масок невуса (BiRefNet)

**Цель:** один раз прогнать пайплайн Насти (детекция RT-DETR-L -> BiRefNet, без удаления волос на всех изображениях из Derm7pt и ISIC Task 2 и сохранить маски невуса как PNG в Drive.

**Схема:** оригинал -> RT-DETR-L (детекция родинки) -> кроп с паддингом -> BiRefNet (320×320) -> ресайз до 300×300 -> PNG.

**Идемпотентно:** если маска уже посчитана, то пропускается.

### 1. Проверка окружения и монтирование Drive

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

### 2. Установка зависимостей

In [ ]:
!pip install -q -U ultralytics transformers segmentation-models-pytorch timm einops kornia

### 3. Проверка наличия весов и скрипта Насти


In [ ]:
from pathlib import Path

WEIGHTS_DIR = Path('/content/drive/MyDrive/Диплом/pipeline_weights')
NASTYA_SCRIPT = WEIGHTS_DIR / 'simple_isic_full_inference.py'
RTDETR_WEIGHTS = WEIGHTS_DIR / 'detector_rtdetr_l_img256_best.pt'
BIREFNET_WEIGHTS = WEIGHTS_DIR / 'best_model_birefnet_320.pt'

required = {
    'Скрипт Насти':       NASTYA_SCRIPT,
    'Веса RT-DETR-L':     RTDETR_WEIGHTS,
    'Веса BiRefNet':      BIREFNET_WEIGHTS,
}
missing = []
for label, path in required.items():
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        print(f'✓ {label:<22}  {path.name}  ({size_mb:.1f} МБ)')
    else:
        missing.append((label, path))
        print(f'✗ {label:<22}  НЕ НАЙДЕН: {path}')

if missing:
    raise FileNotFoundError(
        f'Не хватает {len(missing)} файлов. '
        f'Положи их в {WEIGHTS_DIR} и перезапусти ячейку.'
    )

### 4. Подключаем скрипт Насти как модуль

In [ ]:
import sys
if str(WEIGHTS_DIR) not in sys.path:
    sys.path.insert(0, str(WEIGHTS_DIR))

from simple_isic_full_inference import (
    load_detector, load_birefnet,
    Detection, expand_and_clip_box,
    pil_to_tensor_01, imagenet_normalize, forward_logits,
    IMG_SIZE, DETECTION_CONF, DETECTION_NMS_IOU, DETECTION_CROP_PAD,
    LESION_THRESHOLD, RTDETR_DET_IMGSZ,
    DEVICE, DEVICE_ID,
)

print(f'✓ Импорт OK')
print(f'  IMG_SIZE (BiRefNet):     {IMG_SIZE}')
print(f'  RT-DETR imgsz:           {RTDETR_DET_IMGSZ}')
print(f'  DETECTION_CONF:          {DETECTION_CONF}')
print(f'  DETECTION_CROP_PAD:      {DETECTION_CROP_PAD}')
print(f'  LESION_THRESHOLD:        {LESION_THRESHOLD}')
print(f'  DEVICE:                  {DEVICE}')

### 5. Распаковка датасетов (Derm7pt + ISIC)


In [ ]:
# Derm7pt
os.makedirs('/content/dataset', exist_ok=True)
if not os.path.exists('/content/dataset/release_v0'):
    os.system('unzip -q "/content/drive/MyDrive/Диплом/практика_преддипломная/derm7pt.zip" '
              '-d "/content/dataset"')
    print('✓ Derm7pt распакован')
else:
    print('✓ Derm7pt уже распакован')

# ISIC изображения
os.makedirs('/content/isic', exist_ok=True)
if not os.path.exists('/content/isic/images'):
    os.system('wget -q --show-progress -O /content/isic/images.zip '
              'https://isic-challenge-data.s3.amazonaws.com/2018/'
              'ISIC2018_Task1-2_Training_Input.zip')
    os.system('unzip -q /content/isic/images.zip -d /content/isic/tmp')
    os.system('mv "/content/isic/tmp/ISIC2018_Task1-2_Training_Input" /content/isic/images')
    os.system('rm -rf /content/isic/images.zip /content/isic/tmp')
    print('✓ ISIC images готовы')
else:
    print('✓ ISIC images уже готовы')

### 6. Загрузка двух моделей: RT-DETR-L и BiRefNet

In [ ]:
import time

t0 = time.time()
print('Загружаю RT-DETR-L...')
detector = load_detector('rtdetr', RTDETR_WEIGHTS)
print(f'  ✓ за {time.time()-t0:.1f} с')

t0 = time.time()
print('Загружаю BiRefNet (844 МБ, скачивает архитектуру с HuggingFace)...')
birefnet = load_birefnet(BIREFNET_WEIGHTS)
print(f'  ✓ за {time.time()-t0:.1f} с')

if torch.cuda.is_available():
    print(f'\nVRAM занято: {torch.cuda.memory_allocated()/1e9:.2f} ГБ')

### 7. Функции инференса (детекция -> BiRefNet)

In [ ]:
import numpy as np
from PIL import Image

@torch.no_grad()
def detect_lesion(image_path: Path, image: Image.Image) -> Detection:
    """Детекция самой большой/уверенной родинки на изображении.
    Если детектор ничего не нашёл — fallback на полное изображение.
    """
    width, height = image.size
    results = detector.predict(
        source=str(image_path),
        imgsz=RTDETR_DET_IMGSZ,
        conf=DETECTION_CONF,
        iou=DETECTION_NMS_IOU,
        batch=1,
        device=DEVICE_ID,
        verbose=False,
        stream=False,
        half=torch.cuda.is_available(),
        max_det=20,
    )
    result = results[0]
    if result.boxes is None or len(result.boxes) == 0:
        # Fallback: вся картинка целиком
        return Detection((0, 0, width, height), 0.0, 'rtdetr', used_fallback=True)
    scores = result.boxes.conf.detach().cpu().float().numpy()
    best_idx = int(np.argmax(scores))
    raw_box = result.boxes.xyxy[best_idx].detach().cpu().float().numpy().tolist()
    box = expand_and_clip_box(tuple(raw_box), width, height, DETECTION_CROP_PAD)
    return Detection(box, float(scores[best_idx]), 'rtdetr', used_fallback=False)


@torch.no_grad()
def segment_lesion_on_crop(crop: Image.Image) -> np.ndarray:
    """BiRefNet на кропе (без удаления волос). Возвращает бинарную маску 320×320, uint8 {0,255}."""
    image_01 = pil_to_tensor_01(crop, IMG_SIZE).to(DEVICE)         # (1,3,320,320), [0,1]
    image_norm = imagenet_normalize(image_01)
    model_dtype = next(birefnet.parameters()).dtype
    logits = forward_logits(
        birefnet,
        image_norm.to(dtype=model_dtype),
        target_hw=(IMG_SIZE, IMG_SIZE),
    )
    prob = torch.sigmoid(logits.float())[0, 0]                     # (320,320)
    mask = (prob > LESION_THRESHOLD).cpu().numpy().astype(np.uint8) * 255
    return mask


def predict_lesion_mask_for_full_image(image_path: Path) -> tuple[np.ndarray, dict]:
    """Полный мини-пайплайн A3 для одного изображения.

    Возвращает:
        mask_full_uint8: бинарная маска (H, W) для оригинального изображения, {0,255}
        meta: словарь с диагностической информацией (бокс, скор, fallback и т.д.)
    """
    image = Image.open(image_path).convert('RGB')
    orig_w, orig_h = image.size

    # 1. Детекция
    det = detect_lesion(image_path, image)
    x1, y1, x2, y2 = det.box_xyxy
    crop = image.crop((x1, y1, x2, y2))

    # 2. BiRefNet на кропе
    mask_crop_320 = segment_lesion_on_crop(crop) # (320, 320), uint8

    # 3. Возвращаем маску в координаты исходного изображения
    crop_w = x2 - x1
    crop_h = y2 - y1
    mask_crop = Image.fromarray(mask_crop_320, mode='L').resize(
        (crop_w, crop_h), Image.NEAREST
    )
    mask_full = Image.new('L', (orig_w, orig_h), 0)
    mask_full.paste(mask_crop, (x1, y1))
    mask_full_uint8 = np.asarray(mask_full, dtype=np.uint8)

    meta = {
        'box_xyxy': list(det.box_xyxy),
        'det_score': det.score,
        'used_fallback': det.used_fallback,
        'orig_size': (orig_w, orig_h),
        'positive_pixels': int((mask_full_uint8 > 127).sum()),
        'positive_fraction': float((mask_full_uint8 > 127).mean()),
    }
    return mask_full_uint8, meta

### 8. Sanity-check на одном изображении

Прежде чем запускать на 3000+ файлах, мы проверим, что пайплайн работает и маска выглядит адекватно. Если тут что-то не так, нет смысла гонять всю выборку.

In [ ]:
import glob
import matplotlib.pyplot as plt

# Берём первое изображение из ISIC для проверки
test_images = sorted(glob.glob('/content/isic/images/ISIC_*.jpg'))[:1]
assert test_images, 'Не нашлось ни одного ISIC изображения'

test_path = Path(test_images[0])
print(f'Тестируем на: {test_path.name}')

t0 = time.time()
mask, meta = predict_lesion_mask_for_full_image(test_path)
dt = time.time() - t0
print(f'Время: {dt:.2f} с')
print(f'Бокс: {meta["box_xyxy"]} (score={meta["det_score"]:.3f}, '
      f'fallback={meta["used_fallback"]})')
print(f'Размер маски: {mask.shape}')
print(f'Доля положительных пикселей: {meta["positive_fraction"]*100:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
img = Image.open(test_path).convert('RGB')
axes[0].imshow(img); axes[0].set_title('Оригинал'); axes[0].axis('off')

import matplotlib.patches as mpatches
axes[1].imshow(img)
x1, y1, x2, y2 = meta['box_xyxy']
axes[1].add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                      linewidth=3, edgecolor='cyan', facecolor='none'))
axes[1].set_title(f'RT-DETR (score={meta["det_score"]:.2f})'); axes[1].axis('off')

axes[2].imshow(mask, cmap='gray')
axes[2].set_title(f'BiRefNet маска ({meta["positive_fraction"]*100:.1f}% позитив)')
axes[2].axis('off')
plt.tight_layout(); plt.show()

### 9. Сбор полного списка изображений

In [ ]:
import pandas as pd

DERM_BASE  = '/content/dataset/release_v0'
DERM_IMG   = os.path.join(DERM_BASE, 'images')
DERM_META  = os.path.join(DERM_BASE, 'meta/meta.csv')
ISIC_IMG   = '/content/isic/images'

# Derm7pt: берём из meta.csv
df_meta = pd.read_csv(DERM_META)
all_derm_files = glob.glob(os.path.join(DERM_IMG, '**/*'), recursive=True)
path_map = {f.lower(): f for f in all_derm_files if os.path.isfile(f)}

derm_tasks = []
for _, row in df_meta.iterrows():
    rel = row['derm']
    full_path = path_map.get(os.path.join(DERM_IMG, rel).lower())
    if full_path is None:
        continue
    # Используем basename без расширения как image_id для маски
    image_id = Path(full_path).stem
    derm_tasks.append({'source': 'derm7pt', 'image_id': image_id, 'src_path': full_path})

# ISIC: все jpg
isic_tasks = []
for fname in sorted(os.listdir(ISIC_IMG)):
    if not fname.endswith('.jpg'):
        continue
    image_id = Path(fname).stem
    isic_tasks.append({'source': 'isic', 'image_id': image_id,
                       'src_path': os.path.join(ISIC_IMG, fname)})

all_tasks = derm_tasks + isic_tasks
print(f'Derm7pt: {len(derm_tasks)} изображений')
print(f'ISIC:    {len(isic_tasks)} изображений')
print(f'Всего:   {len(all_tasks)} изображений')

### 10. Папки для масок и метаданных

Маски PNG сохраняем в координатах оригинального изображения (полный размер). Это даст максимальную гибкость в эксп.7, при загрузке датасет сам отресайзит до нужного IMG_SIZE через NEAREST.

In [ ]:
MASKS_ROOT = Path('/content/drive/MyDrive/Диплом/lesion_masks')
(MASKS_ROOT / 'derm7pt').mkdir(parents=True, exist_ok=True)
(MASKS_ROOT / 'isic').mkdir(parents=True, exist_ok=True)
METADATA_CSV = MASKS_ROOT / 'metadata.csv'
print(f'Папка для масок: {MASKS_ROOT}')

def mask_path_for(task):
    return MASKS_ROOT / task['source'] / f"{task['image_id']}.png"

### 11. Главный цикл предрасчёта

**Идемпотентность:** перед обработкой проверяем, существует ли маска. Если да,то пропускаем.

**Что логируем:**
- bbox детектора и его score
- used_fallback (детектор не нашёл, то взяли всё изображение)
- positive_fraction маски (если 0 или 1, то это что-то странное, на это обратим внимание в диагностике)
- elapsed_sec на одно изображение

In [ ]:
from tqdm.auto import tqdm
import json

# Если metadata.csv уже есть - подгружаем, чтобы не терять данные при перезапуске
if METADATA_CSV.exists():
    metadata_df = pd.read_csv(METADATA_CSV)
    print(f'Подгружено {len(metadata_df)} записей из существующего metadata.csv')
    done_ids = set(zip(metadata_df['source'], metadata_df['image_id']))
else:
    metadata_df = pd.DataFrame()
    done_ids = set()

new_records = []
n_skipped = 0
n_processed = 0
n_failed = 0
errors = []

for task in tqdm(all_tasks, desc='BiRefNet inference'):
    key = (task['source'], task['image_id'])
    out_path = mask_path_for(task)

    # Идемпотентность: маска уже есть и запись есть
    if out_path.exists() and key in done_ids:
        n_skipped += 1
        continue

    try:
        t0 = time.time()
        mask, meta = predict_lesion_mask_for_full_image(Path(task['src_path']))
        dt = time.time() - t0

        # Сохраняем маску
        Image.fromarray(mask, mode='L').save(out_path, optimize=True)

        new_records.append({
            'source':            task['source'],
            'image_id':          task['image_id'],
            'mask_path':         str(out_path),
            'src_path':          task['src_path'],
            'box_xyxy':          json.dumps(meta['box_xyxy']),
            'det_score':         meta['det_score'],
            'used_fallback':     meta['used_fallback'],
            'orig_w':            meta['orig_size'][0],
            'orig_h':            meta['orig_size'][1],
            'positive_pixels':   meta['positive_pixels'],
            'positive_fraction': meta['positive_fraction'],
            'elapsed_sec':       dt,
        })
        n_processed += 1
    except Exception as exc:
        n_failed += 1
        errors.append((task['source'], task['image_id'], str(exc)))
        if n_failed <= 5:
            print(f'\n⚠ Ошибка на {task["source"]}/{task["image_id"]}: {exc}')

print(f'\nИтоги:')
print(f'  Обработано:    {n_processed}')
print(f'  Пропущено:     {n_skipped} (уже было)')
print(f'  Ошибки:        {n_failed}')

In [ ]:
# Объединяем со старым CSV и сохраняем
if new_records:
    new_df = pd.DataFrame(new_records)
    metadata_df = pd.concat([metadata_df, new_df], ignore_index=True)
    # На случай если в старом CSV были дубли — оставим последнюю запись для каждого image_id
    metadata_df = metadata_df.drop_duplicates(
        subset=['source', 'image_id'], keep='last'
    ).reset_index(drop=True)
    metadata_df.to_csv(METADATA_CSV, index=False)
    print(f'✓ metadata.csv обновлён: {len(metadata_df)} записей')
else:
    print('Новых записей нет, metadata.csv не изменён')

if errors:
    print(f'\nПолный список ошибок ({len(errors)}):')
    for src, img_id, msg in errors[:20]:
        print(f'  {src}/{img_id}: {msg}')
    if len(errors) > 20:
        print(f'  ...и ещё {len(errors)-20}')

### 12. Диагностика результатов

Прежде чем считать предрасчёт законченным, проверим распределение качества масок:
- Сколько изображений ушло в fallback (детектор не нашёл бокс)?
- Сколько масок подозрительно пустых (доля позитива <1%), значит BiRefNet не нашёл родинку, такие случаи лучше посмотреть глазами.
- Сколько масок подозрительно больших (>80% изображения), обычно артефакт fallback'а или странного входа.

In [ ]:
if not metadata_df.empty:
    print(f'Всего масок: {len(metadata_df)}')
    print(f'  Derm7pt: {(metadata_df["source"]=="derm7pt").sum()}')
    print(f'  ISIC:    {(metadata_df["source"]=="isic").sum()}')

    fb = metadata_df['used_fallback'].sum()
    print(f'\nFallback (детектор не нашёл бокс): {fb} ({100*fb/len(metadata_df):.1f}%)')

    empty = (metadata_df['positive_fraction'] < 0.01).sum()
    huge  = (metadata_df['positive_fraction'] > 0.80).sum()
    print(f'Подозрительно пустые (<1% позитива):  {empty}')
    print(f'Подозрительно большие (>80% позитива): {huge}')

    print(f'\nРаспределение позитивной фракции:')
    print(metadata_df['positive_fraction'].describe())

    print(f'\nСреднее время на изображение: {metadata_df["elapsed_sec"].mean():.2f} с')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(metadata_df['positive_fraction'], bins=50, color='#4C72B0', edgecolor='white')
    axes[0].axvline(0.01, color='red', ls='--', label='1%')
    axes[0].axvline(0.80, color='red', ls='--')
    axes[0].set_xlabel('Доля положительных пикселей')
    axes[0].set_ylabel('Кол-во изображений')
    axes[0].set_title('Распределение размера маски невуса')
    axes[0].legend(); axes[0].grid(alpha=.3)

    valid_scores = metadata_df.loc[~metadata_df['used_fallback'], 'det_score']
    axes[1].hist(valid_scores, bins=50, color='#55A868', edgecolor='white')
    axes[1].set_xlabel('Score детектора (RT-DETR-L)')
    axes[1].set_ylabel('Кол-во изображений')
    axes[1].set_title(f'Уверенность детектора (без {fb} fallback)')
    axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

### 13. Визуальная проверка случайных примеров

Покажем 6 случайных изображений с наложенной маской, чтобы убедиться, что BiRefNet действительно выделяет родинку, а не что-то другое.

In [ ]:
import random
random.seed(42)

sample = metadata_df.sample(n=min(6, len(metadata_df)), random_state=42)

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for i, (_, row) in enumerate(sample.iterrows()):
    img = Image.open(row['src_path']).convert('RGB')
    mask = np.array(Image.open(row['mask_path']).convert('L'))

    axes[0, i].imshow(img); axes[0, i].axis('off')
    fb_str = ' [FALLBACK]' if row['used_fallback'] else ''
    axes[0, i].set_title(
        f'{row["source"]}/{row["image_id"][:20]}\n'
        f'score={row["det_score"]:.2f}{fb_str}',
        fontsize=8
    )

    # Наложение маски
    overlay = np.array(img).copy()
    red = np.zeros_like(overlay); red[..., 0] = 255
    alpha = 0.4 * (mask[..., None] > 127)
    overlay = (overlay * (1 - alpha) + red * alpha).astype(np.uint8)
    axes[1, i].imshow(overlay); axes[1, i].axis('off')
    axes[1, i].set_title(f'позитив={row["positive_fraction"]*100:.1f}%', fontsize=8)

plt.suptitle('Случайные примеры: оригинал (верх) + маска невуса (низ, красным)', fontsize=11)
plt.tight_layout(); plt.show()

### 14. Проверка подозрительных случаев


In [ ]:
suspicious = metadata_df[
    (metadata_df['positive_fraction'] < 0.01) |
    (metadata_df['positive_fraction'] > 0.80) |
    (metadata_df['used_fallback'])
]
print(f'Подозрительных случаев: {len(suspicious)}')

if len(suspicious) > 0:
    # Покажем до 6 первых
    to_show = suspicious.head(6)
    n = len(to_show)
    fig, axes = plt.subplots(2, n, figsize=(3.5*n, 7))
    if n == 1:
        axes = axes.reshape(2, 1)
    for i, (_, row) in enumerate(to_show.iterrows()):
        img = Image.open(row['src_path']).convert('RGB')
        mask = np.array(Image.open(row['mask_path']).convert('L'))
        axes[0, i].imshow(img); axes[0, i].axis('off')
        flags = []
        if row['used_fallback']: flags.append('FB')
        if row['positive_fraction'] < 0.01: flags.append('EMPTY')
        if row['positive_fraction'] > 0.80: flags.append('HUGE')
        axes[0, i].set_title(
            f'{row["image_id"][:20]}\n{"|".join(flags)} pos={row["positive_fraction"]*100:.1f}%',
            fontsize=8
        )
        axes[1, i].imshow(mask, cmap='gray'); axes[1, i].axis('off')
    plt.suptitle('Подозрительные маски: ручная проверка нужна', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('✓ Подозрительных случаев нет, всё в порядке')


Маски невуса лежат в:
- `/content/drive/MyDrive/Диплом/lesion_masks/derm7pt/<image_id>.png`
- `/content/drive/MyDrive/Диплом/lesion_masks/isic/<image_id>.png`

Метаданные (бокс, скор, доля позитива) — в `/content/drive/MyDrive/Диплом/lesion_masks/metadata.csv`.

В эксп.7 датасет будет открывать маски как обычные PNG и ресайзить до `IMG_SIZE=300` через `INTER_NEAREST`. Без обращения к BiRefNet, ML-моделям или Drive больше одного раза.


In [ ]:
import pandas as pd
from pathlib import Path

METADATA_CSV = Path('/content/drive/MyDrive/Диплом/lesion_masks/metadata.csv')
df = pd.read_csv(METADATA_CSV)

print('=== Fallback по источникам ===')
fb = df.groupby('source')['used_fallback'].agg(['sum', 'count', 'mean'])
fb.columns = ['fallback', 'total', 'fb_rate']
print(fb)

print('\n=== Distribution det_score для не-fallback ===')
ok = df[~df['used_fallback']]
print(ok.groupby('source')['det_score'].describe())

print('\n=== Размеры изображений: fallback vs не-fallback ===')
df['mp'] = df['orig_w'] * df['orig_h'] / 1e6  # мегапиксели
print(df.groupby('used_fallback')['mp'].describe())

print('\n=== Распределение по доле позитива у fallback vs не-fallback ===')
print(df.groupby('used_fallback')['positive_fraction'].describe())